In [1]:

import sys
sys.path.insert(0, '/srv/python_packages')
import mlflow

try:
    mlflow.set_tracking_uri('http://localhost:6000')
    mlflow.set_experiment('latam2cloud')
    MLFLOW_OK = True
    print('MLflow OK — http://localhost:6000  |  experimento: latam2cloud')
except Exception as _e:
    MLFLOW_OK = False
    print(f'MLflow no disponible ({_e}) — continúa sin tracking')


MLflow OK — http://localhost:6000  |  experimento: latam2cloud


In [2]:

import os, sys, time, pickle, warnings, gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (cohen_kappa_score, f1_score,
                             balanced_accuracy_score, confusion_matrix)
warnings.filterwarnings('ignore')

SAVE_DIR = '/srv/work'
IN_FEAT  = f'{SAVE_DIR}/te_features_v6.parquet'
OUT_DIR  = f'{SAVE_DIR}/models'
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
HORIZON      = 2
LABEL_COLS   = {0: 'Normal', 1: 'Baja', 2: 'Media', 3: 'Alta'}
ROUTE_KEYS   = ['source_agent', 'cloud_provider', 'region']
ID_COLS      = ROUTE_KEYS + ['window']
TARGET_COL   = 'label_ventana'
SPLIT_COL    = 'split'

UMBRALES = {
    'qwk':         {'floor': 0.65, 'target': 0.72, 'stretch': 0.80},
    'recall_alta': {'floor': 0.55, 'target': 0.75, 'stretch': 0.85},
}

CHECKPOINT_RF = True

print(f'LightGBM {lgb.__version__}')
print(f'Input : {IN_FEAT}')
print(f'Output: {OUT_DIR}')


LightGBM 4.7.0
Input : /srv/work/te_features_v6.parquet
Output: /srv/work/models


In [3]:

print('Cargando te_features_v6.parquet (columnas seleccionadas)...')
schema = pq.read_schema(IN_FEAT)
all_cols = schema.names
keep_cols = [c for c in all_cols
             if not c.startswith('n_samp')]
df = pq.read_table(IN_FEAT, columns=keep_cols).to_pandas()
print(f'  Shape cargado      : {df.shape}')

df['y'] = df.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df = df.dropna(subset=['y'])
df['y'] = df['y'].astype(int)
print(f'  Post target-shift  : {df.shape}')

feat_cols = [c for c in df.columns
             if c not in ID_COLS + [TARGET_COL, SPLIT_COL, 'y']
             and not c.startswith('n_samp')]
print(f'  Features modelo    : {len(feat_cols)}')

ratio_idx = [i for i, c in enumerate(feat_cols) if 'ratio' in c]
other_idx  = [i for i, c in enumerate(feat_cols) if 'ratio' not in c]

train_mask = df[SPLIT_COL] == 'train'
val_mask   = df[SPLIT_COL] == 'val'
test_mask  = df[SPLIT_COL] == 'test'

y_train = df.loc[train_mask, 'y'].values.astype(int)
y_val   = df.loc[val_mask,   'y'].values.astype(int)
y_test  = df.loc[test_mask,  'y'].values.astype(int)

def imputar_nan(X):
    for idx in ratio_idx:
        m = np.isnan(X[:, idx])
        if m.any(): X[m, idx] = 1.0
    for idx in other_idx:
        m = np.isnan(X[:, idx])
        if m.any(): X[m, idx] = 0.0
    return X

print('\nExtrayendo arrays (float32)...')
X_train = imputar_nan(df.loc[train_mask, feat_cols].values.astype('float32'))
del df; gc.collect()
df_val = pq.read_table(IN_FEAT, columns=keep_cols,
                       filters=[('split', '=', 'val')]).to_pandas()
df_val['y'] = df_val.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df_val = df_val.dropna(subset=['y'])
y_val  = df_val['y'].values.astype(int)
X_val  = imputar_nan(df_val[feat_cols].values.astype('float32'))
del df_val; gc.collect()
if CHECKPOINT_RF:
    print(f'  CHECKPOINT_RF=True — y_val val-only: {len(y_val):,} filas (alineado con npy)')

if not CHECKPOINT_RF:
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train).astype('float32')
    X_val_sc   = scaler.transform(X_val).astype('float32')
    with open(f'{OUT_DIR}/scaler_v6.pkl', 'wb') as f:
        pickle.dump(scaler, f)
    print(f'  scaler_v6.pkl guardado')
else:
    X_train_sc = None
    X_val_sc   = None
    print('  CHECKPOINT_RF=True — scaler omitido (RF no requiere X_train_sc)')

print('\nDistribución de clases:')
for split_name, y_arr in [('train', y_train), ('val', y_val)]:
    u, c = np.unique(y_arr, return_counts=True)
    print(f'  {split_name}: ' + '  '.join(
        f'{LABEL_COLS[k]}={v:,} ({100*v/len(y_arr):.1f}%)' for k,v in zip(u,c)))
print(f'\n  Alta train: {(y_train==3).sum():,} ventanas')
print(f'  Alta val  : {(y_val==3).sum():,} ventanas')


Cargando te_features_v6.parquet (columnas seleccionadas)...
  Shape cargado      : (7628801, 106)
  Post target-shift  : (7626731, 107)
  Features modelo    : 100

Extrayendo arrays (float32)...
  CHECKPOINT_RF=True — y_val val-only: 1,138,777 filas (alineado con npy)
  CHECKPOINT_RF=True — scaler omitido (RF no requiere X_train_sc)

Distribución de clases:
  train: Normal=4,621,399 (86.4%)  Baja=509,392 (9.5%)  Media=112,220 (2.1%)  Alta=106,583 (2.0%)
  val: Normal=1,024,150 (89.9%)  Baja=70,518 (6.2%)  Media=22,606 (2.0%)  Alta=21,503 (1.9%)

  Alta train: 106,583 ventanas
  Alta val  : 21,503 ventanas


In [4]:

def evaluar(nombre, y_true, y_prob, verbose=True):
    y_pred   = np.argmax(y_prob, axis=1)
    qwk      = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    bal_acc  = balanced_accuracy_score(y_true, y_pred)
    cm       = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    rec_alta = cm[3,3] / cm[3].sum() if cm[3].sum() > 0 else 0.0
    candidato = qwk >= UMBRALES['qwk']['floor'] and rec_alta >= UMBRALES['recall_alta']['floor']

    if verbose:
        sep = '─' * 80
        flag_qwk = '✅' if qwk      >= UMBRALES['qwk']['floor']         else '❌'
        flag_rec = '✅' if rec_alta >= UMBRALES['recall_alta']['floor']  else '❌'
        print(f'\n{sep}')
        print(f'  {nombre}')
        print(f'{sep}')
        print(f'  QWK      = {qwk:.4f}  {flag_qwk}  (floor≥{UMBRALES["qwk"]["floor"]})')
        print(f'  RecAlta  = {rec_alta:.4f}  {flag_rec}  (floor≥{UMBRALES["recall_alta"]["floor"]})')
        print(f'  MacroF1  = {macro_f1:.4f}')
        print(f'  BalAcc   = {bal_acc:.4f}')
        print(f'  CANDIDATO: {"✅ SÍ" if candidato else "❌ NO"}')
        print(f'  Confusion matrix (Normal/Baja/Media/Alta):')
        for i, row in enumerate(cm):
            print(f'    {LABEL_COLS[i]:6s}: {row}')
    return {'qwk': qwk, 'macro_f1': macro_f1, 'bal_acc': bal_acc,
            'recall_alta': rec_alta, 'candidato': candidato, 'cm': cm}

print('evaluar() definida.')


evaluar() definida.


In [5]:

print('Baseline estático — etiqueta actual como predicción...')

from sklearn.preprocessing import label_binarize
df_tmp = pq.read_table(IN_FEAT, columns=ROUTE_KEYS + [TARGET_COL, SPLIT_COL]).to_pandas()
y_val_noshift = df_tmp.loc[df_tmp[SPLIT_COL]=='val', TARGET_COL].values.astype(int)
n = len(y_val)
y_val_noshift = y_val_noshift[:n]
del df_tmp; gc.collect()

proba_baseline = label_binarize(y_val_noshift, classes=[0,1,2,3]).astype('float32')
res_baseline   = evaluar('Baseline estático (etiqueta actual — sin ML)', y_val, proba_baseline)


Baseline estático — etiqueta actual como predicción...

────────────────────────────────────────────────────────────────────────────────
  Baseline estático (etiqueta actual — sin ML)
────────────────────────────────────────────────────────────────────────────────
  QWK      = 0.1029  ❌  (floor≥0.65)
  RecAlta  = 0.1004  ❌  (floor≥0.55)
  MacroF1  = 0.3205
  BalAcc   = 0.3202
  CANDIDATO: ❌ NO
  Confusion matrix (Normal/Baja/Media/Alta):
    Normal: [935994  52674  17884  17598]
    Baja  : [53044 14292  2258   924]
    Media : [18550  1821  1439   796]
    Alta  : [17916   679   750  2158]


In [6]:

if CHECKPOINT_RF:
    print('CHECKPOINT_RF=True — cargando proba_rf_v10 desde disco...')
    proba_rf_v10 = np.load(f'{OUT_DIR}/rf_v10_val_proba.npy')
    assert proba_rf_v10.shape[0] == len(y_val), (
        f'Mismatch y_val ({len(y_val)}) vs proba ({proba_rf_v10.shape[0]})'
        ' — verificar que CHECKPOINT_RF=True y df completo en Cell 2'
    )
    print(f'  proba_rf_v10: {proba_rf_v10.shape} ✅')
    import joblib
    try:
        rf_v10_tmp = joblib.load(f'{OUT_DIR}/rf_v10_model.pkl')
        fi_values = rf_v10_tmp.feature_importances_.copy()
        del rf_v10_tmp; gc.collect()
        print(f'  fi_values cargado desde rf_v10_model.pkl ✅')
    except Exception as e:
        fi_values = None
        print(f'  fi_values no disponible ({e}) — Cell 9 omitirá MDI')
    res_rf_v10 = evaluar('RF v10 [checkpoint] (proba desde disco)', y_val, proba_rf_v10)
    if MLFLOW_OK:
        with mlflow.start_run(run_name='rf_v10'):
            mlflow.log_params({'checkpoint': True, 'dataset': 'te_features_v6.parquet'})
            mlflow.log_metrics({'qwk_val': res_rf_v10['qwk'],
                                'recall_alta_val': res_rf_v10['recall_alta']})
        print('  MLflow run rf_v10 [checkpoint] registrado')
else:
    print('RF v10 — n_estimators=300, balanced_subsample, min_samples_leaf=5')
    print(f'  X_train_sc: {X_train_sc.shape}  |  y_train Alta: {(y_train==3).sum():,}')
    print('  Tiempo estimado: ~90-150 min (mayor n_Alta vs v9)')

    t0 = time.time()
    rf_v10 = RandomForestClassifier(
        n_estimators     = 300,
        max_features     = 'sqrt',
        min_samples_leaf = 5,
        n_jobs           = 4,
        class_weight     = 'balanced_subsample',
        random_state     = RANDOM_STATE,
        verbose          = 0,
    )
    rf_v10.fit(X_train_sc, y_train)
    elapsed = time.time() - t0
    print(f'  Entrenado en {elapsed/60:.1f} min')

    CHUNK = 50_000
    n_val = X_val_sc.shape[0]
    proba_rf_v10 = np.zeros((n_val, 4), dtype='float32')
    t1 = time.time()
    for i in range(0, n_val, CHUNK):
        proba_rf_v10[i:i+CHUNK] = rf_v10.predict_proba(X_val_sc[i:i+CHUNK])
    print(f'  Predicción val: {time.time()-t1:.1f}s')

    np.save(f'{OUT_DIR}/rf_v10_val_proba.npy', proba_rf_v10)
    print(f'  Guardado: rf_v10_val_proba.npy')

    fi_values = rf_v10.feature_importances_.copy()

    import joblib
    joblib.dump(rf_v10, f'{OUT_DIR}/rf_v10_model.pkl', compress=0)
    print(f'  Guardado: rf_v10_model.pkl (joblib)')

    del rf_v10; gc.collect()
    print('  rf_v10 liberado de RAM (Fix-MEM-06) — fi_values conservado')

    res_rf_v10 = evaluar('RF v10 (300 estimadores, balanced_subsample)', y_val, proba_rf_v10)

    print(f'\n  Δ QWK    vs RF v9 val (0.6503): {res_rf_v10["qwk"]      - 0.6503:+.4f}')
    print(f'  Δ RecAlta vs RF v9 val (0.6526): {res_rf_v10["recall_alta"] - 0.6526:+.4f}')

    if MLFLOW_OK:
        with mlflow.start_run(run_name='rf_v10'):
            mlflow.log_params({'n_estimators': 300, 'class_weight': 'balanced_subsample',
                               'min_samples_leaf': 5, 'max_features': 'sqrt',
                               'dataset': 'te_features_v6.parquet', 'oe': 'OE2'})
            mlflow.log_metrics({'qwk_val': res_rf_v10['qwk'],
                                'recall_alta_val': res_rf_v10['recall_alta'],
                                'macro_f1_val': res_rf_v10['macro_f1'],
                                'bal_acc_val': res_rf_v10['bal_acc']})
            mlflow.log_param('candidato_val', res_rf_v10['candidato'])
        print('  MLflow run rf_v10 registrado')

    del X_train_sc; gc.collect()
    print('  X_train_sc liberado (Fix-MEM-04)')


CHECKPOINT_RF=True — cargando proba_rf_v10 desde disco...
  proba_rf_v10: (1138777, 4) ✅
  fi_values no disponible () — Cell 9 omitirá MDI

────────────────────────────────────────────────────────────────────────────────
  RF v10 [checkpoint] (proba desde disco)
────────────────────────────────────────────────────────────────────────────────
  QWK      = 0.6223  ❌  (floor≥0.65)
  RecAlta  = 0.5073  ❌  (floor≥0.55)
  MacroF1  = 0.6500
  BalAcc   = 0.6434
  CANDIDATO: ❌ NO
  Confusion matrix (Normal/Baja/Media/Alta):
    Normal: [996468  15240   3730   8712]
    Baja  : [15776 50796  2738  1208]
    Media : [7676 2508 8436 3986]
    Alta  : [ 8364   540  1691 10908]
🏃 View run rf_v10 at: http://localhost:6000/#/experiments/681062908553326840/runs/fc47fee29a37470ebc532be16ee2246d
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
  MLflow run rf_v10 [checkpoint] registrado


In [7]:

print('LGB v10 — num_leaves=63, n_estimators=500, sqrt sample_weight')
print(f'  X_train: {X_train.shape}')

from collections import Counter
counts = Counter(y_train.tolist())
n_total   = len(y_train)
n_classes = 4
sw_map    = {k: np.sqrt(n_total / (n_classes * v)) for k, v in counts.items()}
sample_w  = np.array([sw_map[y] for y in y_train], dtype='float32')
print(f'  Pesos sqrt por clase: ' +
      '  '.join(f'{LABEL_COLS[k]}={sw_map[k]:.3f}' for k in sorted(sw_map)))

t0 = time.time()
lgb_v10 = lgb.LGBMClassifier(
    num_leaves      = 63,
    n_estimators    = 500,
    learning_rate   = 0.05,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    min_child_samples=20,
    n_jobs          = -1,
    random_state    = RANDOM_STATE,
    verbose         = -1,
)
lgb_v10.fit(X_train, y_train, sample_weight=sample_w)
elapsed = time.time() - t0
print(f'  Entrenado en {elapsed/60:.1f} min')

proba_lgb_v10 = lgb_v10.predict_proba(X_val).astype('float32')
np.save(f'{OUT_DIR}/lgb_v10_val_proba.npy', proba_lgb_v10)
lgb_v10.booster_.save_model(f'{OUT_DIR}/lgb_v10_model.txt')
print(f'  Guardado: lgb_v10_model.txt + lgb_v10_val_proba.npy')

res_lgb_v10 = evaluar('LGB v10 (num_leaves=63, 500 iter, sqrt weight)', y_val, proba_lgb_v10)

print(f'\n  Δ QWK    vs LGB v7 val (0.6300): {res_lgb_v10["qwk"]       - 0.6300:+.4f}')
print(f'  Δ RecAlta vs LGB v7 val (0.6824): {res_lgb_v10["recall_alta"] - 0.6824:+.4f}')

del X_train, X_val, X_val_sc; gc.collect()
print('  X_train / X_val / X_val_sc liberados (Fix-MEM-07) — ~4.6 GB')

if MLFLOW_OK:
    with mlflow.start_run(run_name='lgb_v10'):
        mlflow.log_params({'num_leaves': 63, 'n_estimators': 500,
                           'learning_rate': 0.05, 'sample_weight': 'sqrt',
                           'dataset': 'te_features_v6.parquet', 'oe': 'OE2'})
        mlflow.log_metrics({'qwk_val': res_lgb_v10['qwk'],
                            'recall_alta_val': res_lgb_v10['recall_alta'],
                            'macro_f1_val': res_lgb_v10['macro_f1'],
                            'bal_acc_val': res_lgb_v10['bal_acc']})
        mlflow.log_param('candidato_val', res_lgb_v10['candidato'])
    print('  MLflow run lgb_v10 registrado')


LGB v10 — num_leaves=63, n_estimators=500, sqrt sample_weight
  X_train: (5349594, 100)
  Pesos sqrt por clase: Normal=0.538  Baja=1.620  Media=3.452  Alta=3.542
  Entrenado en 11.8 min
  Guardado: lgb_v10_model.txt + lgb_v10_val_proba.npy

────────────────────────────────────────────────────────────────────────────────
  LGB v10 (num_leaves=63, 500 iter, sqrt weight)
────────────────────────────────────────────────────────────────────────────────
  QWK      = 0.5740  ❌  (floor≥0.65)
  RecAlta  = 0.6487  ✅  (floor≥0.55)
  MacroF1  = 0.6375
  BalAcc   = 0.7127
  CANDIDATO: ❌ NO
  Confusion matrix (Normal/Baja/Media/Alta):
    Normal: [967098  23260  12763  21029]
    Baja  : [12239 52905  3950  1424]
    Media : [ 4726  2333 11478  4069]
    Alta  : [ 4604   516  2435 13948]

  Δ QWK    vs LGB v7 val (0.6300): -0.0560
  Δ RecAlta vs LGB v7 val (0.6824): -0.0337
  X_train / X_val / X_val_sc liberados (Fix-MEM-07) — ~4.6 GB
🏃 View run lgb_v10 at: http://localhost:6000/#/experiments/681062

In [8]:

W_RF_D38, W_LGB_D38 = 0.80, 0.20

proba_d38_base = (W_RF_D38 * proba_rf_v10 + W_LGB_D38 * proba_lgb_v10).astype('float32')
res_d38_base   = evaluar(
    f'Ensemble base D38 (RF×{W_RF_D38} + LGB×{W_LGB_D38}) — pesos v9',
    y_val, proba_d38_base
)

print(f'\n  Δ QWK    vs Ensemble D38 v9 val (0.6529): {res_d38_base["qwk"]       - 0.6529:+.4f}')
print(f'  Δ RecAlta vs Ensemble D38 v9 val (0.6509): {res_d38_base["recall_alta"] - 0.6509:+.4f}')



────────────────────────────────────────────────────────────────────────────────
  Ensemble base D38 (RF×0.8 + LGB×0.2) — pesos v9
────────────────────────────────────────────────────────────────────────────────
  QWK      = 0.6224  ❌  (floor≥0.65)
  RecAlta  = 0.5297  ❌  (floor≥0.55)
  MacroF1  = 0.6542
  BalAcc   = 0.6572
  CANDIDATO: ❌ NO
  Confusion matrix (Normal/Baja/Media/Alta):
    Normal: [993399  16358   4405   9988]
    Baja  : [15067 51321  2908  1222]
    Media : [7224 2450 9074 3858]
    Alta  : [ 7757   523  1832 11391]

  Δ QWK    vs Ensemble D38 v9 val (0.6529): -0.0305
  Δ RecAlta vs Ensemble D38 v9 val (0.6509): -0.1212


In [9]:

print('Grid search pesos ensemble RF/LGB en val')
print('  Espacio: w_RF ∈ [0.50..0.95 paso 0.05]  |  w_LGB = 1 − w_RF')
print('  Criterio: max QWK con candidato (QWK>=0.65 AND RecAlta>=0.55)\n')

resultados = []
for w_rf in np.arange(0.50, 0.96, 0.05):
    w_lgb = round(1.0 - w_rf, 4)
    w_rf  = round(w_rf, 4)
    p     = (w_rf * proba_rf_v10 + w_lgb * proba_lgb_v10)
    y_pred = np.argmax(p, axis=1)
    qwk    = cohen_kappa_score(y_val, y_pred, weights='quadratic')
    cm     = confusion_matrix(y_val, y_pred, labels=[0,1,2,3])
    rec_a  = cm[3,3] / cm[3].sum() if cm[3].sum() > 0 else 0.0
    cand   = qwk >= 0.65 and rec_a >= 0.55
    resultados.append({'w_rf': w_rf, 'w_lgb': w_lgb, 'qwk': qwk,
                        'recall_alta': rec_a, 'candidato': cand})

df_grid = pd.DataFrame(resultados).sort_values('qwk', ascending=False)
print(f'  {"w_RF":>6}  {"w_LGB":>6}  {"QWK":>7}  {"RecAlta":>9}  CAND')
print('  ' + '─'*50)
for _, row in df_grid.iterrows():
    flag = '✅' if row['candidato'] else '  '
    print(f'  {row["w_rf"]:>6.2f}  {row["w_lgb"]:>6.2f}  '
          f'{row["qwk"]:>7.4f}  {row["recall_alta"]:>9.4f}  {flag}')

cands = df_grid[df_grid['candidato'] == True]
if len(cands) > 0:
    best = cands.iloc[0]
    print(f'\n  Mejor candidato: w_RF={best["w_rf"]}  w_LGB={best["w_lgb"]}')
    print(f'  QWK={best["qwk"]:.4f}  RecAlta={best["recall_alta"]:.4f}')
else:
    best = df_grid.iloc[0]
    print(f'\n  Sin combinación candidata — mejor por QWK: w_RF={best["w_rf"]}')
    print(f'  QWK={best["qwk"]:.4f}  RecAlta={best["recall_alta"]:.4f}')

W_RF_OPT  = best['w_rf']
W_LGB_OPT = best['w_lgb']

proba_ens_v10 = (W_RF_OPT * proba_rf_v10 + W_LGB_OPT * proba_lgb_v10).astype('float32')
res_ens_v10   = evaluar(
    f'Ensemble v10 (RF×{W_RF_OPT} + LGB×{W_LGB_OPT}) — óptimo val',
    y_val, proba_ens_v10
)


Grid search pesos ensemble RF/LGB en val
  Espacio: w_RF ∈ [0.50..0.95 paso 0.05]  |  w_LGB = 1 − w_RF
  Criterio: max QWK con candidato (QWK>=0.65 AND RecAlta>=0.55)

    w_RF   w_LGB      QWK    RecAlta  CAND
  ──────────────────────────────────────────────────
    0.85    0.15   0.6227     0.5237    
    0.90    0.10   0.6226     0.5182    
    0.80    0.20   0.6224     0.5297    
    0.95    0.05   0.6224     0.5130    
    0.70    0.30   0.6221     0.5423    
    0.75    0.25   0.6219     0.5349    
    0.65    0.35   0.6217     0.5495    
    0.60    0.40   0.6203     0.5569    
    0.55    0.45   0.6188     0.5656    
    0.50    0.50   0.6159     0.5748    

  Sin combinación candidata — mejor por QWK: w_RF=0.85
  QWK=0.6227  RecAlta=0.5237

────────────────────────────────────────────────────────────────────────────────
  Ensemble v10 (RF×0.85 + LGB×0.15) — óptimo val
────────────────────────────────────────────────────────────────────────────────
  QWK      = 0.6227  ❌  (floo

In [10]:

if fi_values is None:
    print('fi_values no disponible (checkpoint sin pkl) — MDI omitido')
    importancias = pd.Series(np.zeros(len(feat_cols)), index=feat_cols)
else:
    importancias = pd.Series(fi_values, index=feat_cols)
top20 = importancias.nlargest(20)

print('Top 20 features por MDI (RF v10):')
print(f'  {"Rank":>4}  {"Feature":45s}  {"MDI":>8}  {"% acum":>7}')
print('  ' + '─'*70)
acum = 0.0
for rank, (feat, imp) in enumerate(top20.items(), 1):
    acum += imp
    print(f'  {rank:>4}  {feat:45s}  {imp:>8.5f}  {100*acum:>6.1f}%')

print(f'\n  Suma top-20: {top20.sum():.4f} ({100*top20.sum():.1f}% de importancia total)')

top20.to_csv(f'{OUT_DIR}/rf_v10_feature_importance_top20.csv', header=['mdi'])
print(f'  Guardado: rf_v10_feature_importance_top20.csv')


fi_values no disponible (checkpoint sin pkl) — MDI omitido
Top 20 features por MDI (RF v10):
  Rank  Feature                                             MDI   % acum
  ──────────────────────────────────────────────────────────────────────
     1  delta_mad_jitter                                0.00000     0.0%
     2  delta_mad_lat                                   0.00000     0.0%
     3  delta_mad_loss                                  0.00000     0.0%
     4  delta_p95_jitter                                0.00000     0.0%
     5  delta_p95_lat                                   0.00000     0.0%
     6  delta_p95_loss                                  0.00000     0.0%
     7  mad_jitter                                      0.00000     0.0%
     8  mad_lat                                         0.00000     0.0%
     9  mad_loss                                        0.00000     0.0%
    10  mean_jitter                                     0.00000     0.0%
    11  mean_lat               

In [11]:

np.save(f'{OUT_DIR}/ens_v10_val_proba.npy', proba_ens_v10)
print(f'ens_v10_val_proba.npy guardado')
print(f'Candidato congelado: RF×{W_RF_OPT} + LGB×{W_LGB_OPT}  '
      f'QWK={res_ens_v10["qwk"]:.4f}  RecAlta={res_ens_v10["recall_alta"]:.4f}')

if MLFLOW_OK:
    with mlflow.start_run(run_name='ensemble_v10'):
        mlflow.log_params({
            'notebook': '05 modelado v10.ipynb',
            'oe': 'OE2',
            'dataset': 'te_features_v6.parquet',
            'w_rf': W_RF_OPT,
            'w_lgb': W_LGB_OPT,
            'rf_n_estimators': 300,
            'rf_class_weight': 'balanced_subsample',
            'lgb_num_leaves': 63,
            'lgb_n_estimators': 500,
            'lgb_sample_weight': 'sqrt',
            'log_decision': 'LOG-43',
        })
        mlflow.log_metrics({
            'qwk_val':          res_ens_v10['qwk'],
            'recall_alta_val':  res_ens_v10['recall_alta'],
            'macro_f1_val':     res_ens_v10['macro_f1'],
            'bal_acc_val':      res_ens_v10['bal_acc'],
            'qwk_rf_val':       res_rf_v10['qwk'],
            'qwk_lgb_val':      res_lgb_v10['qwk'],
            'baseline_qwk_val': res_baseline['qwk'],
        })
        mlflow.set_tags({
            'candidato_val': str(res_ens_v10['candidato']),
            'parquet_entrada': 'te_features_v6.parquet',
            'parquet_proba': 'ens_v10_val_proba.npy',
        })
    print('MLflow run ensemble_v10 registrado')

sep = '=' * 100
print(f'\n{sep}')
print('  TABLA COMPARATIVA v9 → v10 (validación)')
print(sep)
print(f'  {"Modelo":50s}  {"QWK":>7}  {"RecAlta":>9}  {"MacroF1":>8}  {"Cand":>5}')
print('  ' + '─'*90)

refs = [
    ('Baseline estático v10 (etiqueta actual)',       res_baseline),
    ('RF v10 (300 est, balanced_sub, dataset 162d)',  res_rf_v10),
    ('LGB v10 (63 leaves, 500 iter, sqrt w)',         res_lgb_v10),
    (f'Ensemble base D38 (RF×0.80+LGB×0.20) v9→v10', res_d38_base),
    (f'Ensemble v10 óptimo (RF×{W_RF_OPT}+LGB×{W_LGB_OPT})', res_ens_v10),
]
hist = [
    ('── REF v9 (dataset 83d) ── RF v5.1',    0.6503, 0.6526, None, False),
    ('── REF v9 (dataset 83d) ── Ens D38',    0.6529, 0.6509, None, True),
]

for nombre, res in refs:
    flag = '✅' if res['candidato'] else '❌'
    print(f'  {nombre:50s}  {res["qwk"]:>7.4f}  {res["recall_alta"]:>9.4f}  '
          f'{res["macro_f1"]:>8.4f}  {flag:>5}')

print('  ' + '─'*90)
for nombre, qwk, rec, _, cand in hist:
    flag = '✅' if cand else '──'
    print(f'  {nombre:50s}  {qwk:>7.4f}  {rec:>9.4f}  {"──":>8}  {flag:>5}')

print(f'\n  Umbral candidato: QWK≥0.65 AND RecAlta≥0.55  (Benchmarking Criterios Éxito v2.0)')
print(f'  CANDIDATO FINAL v10: Ensemble RF×{W_RF_OPT} + LGB×{W_LGB_OPT}')
print(f'  → Próximo paso: 06 evaluacion test set v7.ipynb (ONE-SHOT — Sep 8-30)')


ens_v10_val_proba.npy guardado
Candidato congelado: RF×0.85 + LGB×0.15  QWK=0.6227  RecAlta=0.5237
🏃 View run ensemble_v10 at: http://localhost:6000/#/experiments/681062908553326840/runs/36ac98ff54074988b8b62867e54fa8ce
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
MLflow run ensemble_v10 registrado

  TABLA COMPARATIVA v9 → v10 (validación)
  Modelo                                                  QWK    RecAlta   MacroF1   Cand
  ──────────────────────────────────────────────────────────────────────────────────────────
  Baseline estático v10 (etiqueta actual)              0.1029     0.1004    0.3205      ❌
  RF v10 (300 est, balanced_sub, dataset 162d)         0.6223     0.5073    0.6500      ❌
  LGB v10 (63 leaves, 500 iter, sqrt w)                0.5740     0.6487    0.6375      ❌
  Ensemble base D38 (RF×0.80+LGB×0.20) v9→v10          0.6224     0.5297    0.6542      ❌
  Ensemble v10 óptimo (RF×0.85+LGB×0.15)               0.6227     0.5237    0.6534